In [11]:
#based on https://towardsdatascience.com/introduction-to-nlp-part-3-tf-idf-explained-cedb1fc1f7dc
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
import pandas as pd
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [13]:
nltk.download('omw-1.4')
!python -m spacy download es_core_news_sm
nlp = spacy.load("es_core_news_sm")

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 88.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Text preprocessing

In [18]:
#Documents
d1 = 'Puede ser que esté repitiendo el curso, porque siempre lo veo jugando'
d2 = 'Yo no creo que esté repitiendo el curso, porque Emmanuel dijo que eran los mismos proyectos'
print("d1: ", d1)
print("d2: ", d2)

def preprocess_text(document):
    """
    Preprocess an entire document
    """
    processed_doc = nlp(document)

    # Lowercase, lemmatise, and keep only alphabetic tokens (removes punctuation)
    lemmas = [token.lemma_ for token in processed_doc if token.is_alpha]

    # Remove stopwords, terms
    keywords= [lemma for lemma in lemmas if lemma not in stopwords.words('spanish')]
    return keywords

#preprocess the documents
d1_preprocessed = preprocess_text(d1)
d2_preprocessed = preprocess_text(d2)
print("d1 preprocessed: ", d1_preprocessed)
print("d2 preprocessed: ", d2_preprocessed)

d1:  Puede ser que esté repitiendo el curso, porque siempre lo veo jugando
d2:  Yo no creo que esté repitiendo el curso, porque Emmanuel dijo que eran los mismos proyectos
d1 preprocessed:  ['poder', 'ser', 'repetir', 'curso', 'siempre', 'ver', 'jugar']
d2 preprocessed:  ['creer', 'repetir', 'curso', 'Emmanuel', 'decir', 'ser', 'mismo', 'proyecto']


# TF-IDF transformation

In [19]:

def display_tfidfs(X_train_vectorised):
    # Convert sparse matrix to dataframe
    X_train = pd.DataFrame.sparse.from_spmatrix(X_train_vectorised)
    print("sparse X_train")
    print(X_train)
    # Save mapping on which index refers to which words
    #col_map = {v:k for k, v in X_train_vectorised.vocabulary_.items()}
    # Rename each column using the mapping
    #for col in X_train.columns:
        #X_train.rename(columns={col: col_map[col]}, inplace=True)
    #print(X_train)



# Create an instance of TfidfVectorizer
# we can send the preprocessing function as part of the tfidf vectoriser
tfidf_vectoriser = TfidfVectorizer(analyzer = preprocess_text)
# Create dataframe, input of the Tfidf vectoriser
X_train = pd.DataFrame({'corpus': [d1, d2]})


# Vectorise the data using the TF-IDF
#The result is encoded in a sparse matrix (i.e, 0 values are not included)
X_train_vectorised = tfidf_vectoriser.fit_transform(X_train['corpus'])
print(X_train_vectorised)
display_tfidfs(X_train_vectorised)




<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 15 stored elements and shape (2, 12)>
  Coords	Values
  (0, 6)	0.42567716283146345
  (0, 9)	0.3028728072833121
  (0, 8)	0.3028728072833121
  (0, 2)	0.3028728072833121
  (0, 10)	0.42567716283146345
  (0, 11)	0.42567716283146345
  (0, 4)	0.42567716283146345
  (1, 9)	0.27867523200844097
  (1, 8)	0.27867523200844097
  (1, 2)	0.27867523200844097
  (1, 1)	0.3916683150818112
  (1, 0)	0.3916683150818112
  (1, 3)	0.3916683150818112
  (1, 5)	0.3916683150818112
  (1, 7)	0.3916683150818112
sparse X_train
         0         1         2         3         4         5         6   \
0         0         0  0.302873         0  0.425677         0  0.425677   
1  0.391668  0.391668  0.278675  0.391668         0  0.391668         0   

         7         8         9         10        11  
0         0  0.302873  0.302873  0.425677  0.425677  
1  0.391668  0.278675  0.278675         0         0  
